In [1]:
# Colab bootstrap —— 在 Colab 上第一件事就是跑這格。本機開發時它會自動跳過安裝。
#
# --no-deps        Colab 預裝的 torch 是對著它自己的 CUDA 編的，不能讓 pip 重裝
# --force-reinstall 版本號沒變時 pip 會跳過安裝，推了修正之後要靠它抓到新版
#
# 重裝之後舊模組還留在 sys.modules 裡，**要重啟 kernel** 才吃得到新版。
#
# 用 subprocess 而不是 %pip：這樣同一格在本機與 Colab 都能原樣執行。
import subprocess
import sys

REPO = "git+https://github.com/318amne-Sia/tfn-pytorch.git@main"

# 用 try/import 而不是 importlib.util.find_spec("google.colab")：後者在沒有
# google 這個套件的環境（例如本機）會直接拋 ModuleNotFoundError，不是回 None。
try:
    import google.colab  # noqa: F401

    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", REPO],
        check=True,
    )

# 這個 import 刻意放在安裝之後（E402）：Colab 上要先裝好才 import 得到
import tfn  # noqa: E402

print("tfn", tfn.__version__)

tfn 0.1.0


# 實驗一：3D Tetris 形狀分類

論文 [Tensor Field Networks](https://arxiv.org/abs/1802.08219) §5.1 的復現。

主張是這樣的：**訓練時只餵單一朝向、完全不做旋轉資料增強**，測試時餵隨機旋轉
且平移過的同一批形狀，仍然全部分對。

這不是因為模型「學會」了旋轉，而是因為它根本不需要學——網路內部的表示會跟著
座標一起轉，所以一個朝向學到的東西自動適用於所有朝向。

上游原始碼（TensorFlow 1.x）保留在 `reference/`，可逐行對照。

In [2]:
import numpy as np
import torch

from tfn.shape_classification import (
    SHAPE_NAMES,
    ShapeClassifier,
    evaluate,
    random_pose,
    tetris_shapes,
    train,
)
from tfn.utils import distance_matrix

SEED = 0

shapes = tetris_shapes()
for label, (name, shape) in enumerate(zip(SHAPE_NAMES, shapes, strict=True)):
    print(f"{label}  {name:16s} {[tuple(int(v) for v in p) for p in shape.tolist()]}")

0  chiral_shape_1   [(0, 0, 0), (0, 0, 1), (1, 0, 0), (1, 1, 0)]
1  chiral_shape_2   [(0, 0, 0), (0, 0, 1), (1, 0, 0), (1, -1, 0)]
2  square           [(0, 0, 0), (1, 0, 0), (0, 1, 0), (1, 1, 0)]
3  line             [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 0, 3)]
4  corner           [(0, 0, 0), (0, 0, 1), (0, 1, 0), (1, 0, 0)]
5  T                [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 0)]
6  zigzag           [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 1)]
7  L                [(0, 0, 0), (1, 0, 0), (1, 1, 0), (2, 1, 0)]


## 為什麼這個資料集不好對付

前兩個形狀 `chiral_shape_1` 與 `chiral_shape_2` 是彼此的鏡像。它們的**點對距離
集合完全相同**——只看距離的模型（SchNet）拿到的輸入一模一樣，不可能分開；
只再加上鍵角也不夠（ANI-1）。

要分辨它們，表示裡必須帶有手性資訊。

In [3]:
def pair_distances(shape):
    """6 個點對距離，由小到大。

    只取上三角：distance_matrix 的對角線是 sqrt(EPSILON) = 1e-4 而不是 0
    （它走 norm_with_epsilon），那些不是真的距離。
    """
    i, j = torch.triu_indices(len(shape), len(shape), offset=1)
    return sorted(round(v, 4) for v in distance_matrix(shape)[i, j].tolist())


print("chiral_shape_1 距離:", pair_distances(shapes[0]))
print("chiral_shape_2 距離:", pair_distances(shapes[1]))
print("完全相同:", pair_distances(shapes[0]) == pair_distances(shapes[1]))

chiral_shape_1 距離: [1.0, 1.0, 1.0, 1.4142, 1.4142, 1.7321]
chiral_shape_2 距離: [1.0, 1.0, 1.0, 1.4142, 1.4142, 1.7321]
完全相同: True


## 等變性是結構帶來的，不是學來的

先驗一件事：**還沒訓練**的模型，對隨機旋轉且平移過的輸入就已經吐出相同的
logits。如果這裡不成立，後面訓練得再好都不算數。

In [4]:
torch.manual_seed(SEED)
untrained = ShapeClassifier(len(SHAPE_NAMES))

rng = np.random.default_rng(1)
worst = max(
    (untrained(random_pose(shape, rng)) - untrained(shape)).abs().max().item() for shape in shapes
)
print(f"未訓練模型在旋轉+平移下的 logits 最大偏差：{worst:.2e}")

未訓練模型在旋轉+平移下的 logits 最大偏差：1.30e-07


## 訓練

只餵單一朝向、無資料增強、一次一個形狀（不 batch，同作者）、Adam `lr = 1e-3`。

epoch 數：上游 notebook 用 2001。實測 5 個不同 seed 在 600 epochs 就全部達到
100%，這裡取 1000 留餘裕，本機 CPU 約 20 秒。點雲只有 4 個點、batch = 1，
丟 GPU 只會被 kernel launch 開銷拖慢。

In [5]:
EPOCHS = 1000

torch.manual_seed(SEED)
model = ShapeClassifier(len(SHAPE_NAMES))

history = train(model, shapes, epochs=EPOCHS)

for epoch in range(0, EPOCHS, 100):
    print(f"epoch {epoch:5d}  loss {history[epoch]:.4f}")
print(f"epoch {EPOCHS - 1:5d}  loss {history[-1]:.4f}")

epoch     0  loss 2.0474
epoch   100  loss 0.5824
epoch   200  loss 0.4150
epoch   300  loss 0.2160
epoch   400  loss 0.1735
epoch   500  loss 0.1465
epoch   600  loss 0.0568
epoch   700  loss 0.1057
epoch   800  loss 0.0543
epoch   900  loss 0.0238
epoch   999  loss 0.0102


## 測試：隨機旋轉**且平移**

25 輪 × 8 個形狀 = 200 個樣本，每一個的姿態都是訓練時沒看過的。

（上游 notebook 在這裡有個 bug：算了 `translated_shape` 卻把 `rotated_shape`
餵進去，所以平移那半從來沒被測到。這裡修掉了。）

In [6]:
result = evaluate(model, shapes, rounds=25, rng=SEED)

print(f"樣本數：{sum(len(g) for g in result.predictions.values())}")
print(f"測試準確率：{result.accuracy:.1%}\n")
for name in SHAPE_NAMES:
    print(f"  {name:16s} {result.accuracy_for(name):.1%}")

樣本數：200
測試準確率：100.0%

  chiral_shape_1   100.0%
  chiral_shape_2   100.0%
  square           100.0%
  line             100.0%
  corner           100.0%
  T                100.0%
  zigzag           100.0%
  L                100.0%


## 兩個鏡像形狀

論文對本實驗的主要主張：TFN 不會把這兩個搞混。

In [7]:
for name in ("chiral_shape_1", "chiral_shape_2"):
    guesses = result.predictions[name]
    counts = {g: guesses.count(g) for g in sorted(set(guesses))}
    print(f"{name:16s} 準確率 {result.accuracy_for(name):.1%}  被判成 {counts}")

confused = result.predictions["chiral_shape_1"].count("chiral_shape_2") + result.predictions[
    "chiral_shape_2"
].count("chiral_shape_1")
print(f"\n兩者互相混淆的次數：{confused}")

chiral_shape_1   準確率 100.0%  被判成 {'chiral_shape_1': 25}
chiral_shape_2   準確率 100.0%  被判成 {'chiral_shape_2': 25}

兩者互相混淆的次數：0
